# Chicago taxi trips, 11–17 September 2023

Descriptive summary of one week of **reported** taxi trips (the City notes not every trip is reported, so all figures are reported trips, not total market volume).

## Summary

_(summary placeholder)_

## 1. Setup and data loading

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.width = 160

# Chart style: one hue for magnitude, blue/orange for a two-way comparison (validated reference palette).
BLUE, ORANGE, GREY = '#2a78d6', '#eb6834', '#8a8a86'
plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e6e3', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelsize': 10, 'font.size': 10,
})
thousands = mtick.FuncFormatter(lambda x, _: f'{x:,.0f}')
dollars   = mtick.FuncFormatter(lambda x, _: f'${x:,.0f}')

raw = pd.read_csv('trips.csv', parse_dates=['trip_start_timestamp', 'trip_end_timestamp'])
print(raw.shape)
print('Start range:', raw.trip_start_timestamp.min(), '->', raw.trip_start_timestamp.max())
print('Duplicate trip_id:', raw.trip_id.duplicated().sum(), '| fully duplicated rows:', raw.duplicated().sum())

## 2. Data checks and cleaning

First the checks, then the decisions. Every rule below is applied once, to the whole dataset, and **the same cleaned table (`df`) feeds every question**, so counts and shares are consistent across sections.

In [ ]:
money = ['fare', 'tips', 'tolls', 'extras', 'trip_total']
num = ['trip_seconds', 'trip_miles'] + money
miss = raw.isna().sum()
print('Missing values (columns used in this analysis):')
print(miss[['trip_end_timestamp','trip_seconds','pickup_community_area','dropoff_community_area'] + money + ['payment_type','company','taxi_id']].to_frame('n_missing').assign(pct=lambda d: d.n_missing/len(raw)*100))
print()
print(raw[num].describe(percentiles=[.01, .5, .99, .999]).T)

In [ ]:
# Where are the missing money fields? And what do zero / extreme values look like?
m = raw[raw.fare.isna()]
print('Rows with all five money fields missing:', len(m), '| all five missing together:', raw[money].isna().all(axis=1).sum())
print(m.payment_type.value_counts().to_string()); print()

print('trip_total == 0:', (raw.trip_total == 0).sum(), '| fare == 0:', (raw.fare == 0).sum())
print('trip_total > $500:', (raw.trip_total > 500).sum(), '| tolls > $100:', (raw.tolls > 100).sum(), '| extras > $100:', (raw.extras > 100).sum())
print('Zero miles AND zero seconds:', ((raw.trip_miles == 0) & (raw.trip_seconds.fillna(0) == 0)).sum(),
      '| zero miles (any duration):', (raw.trip_miles == 0).sum(), '| miles > 100:', (raw.trip_miles > 100).sum(),
      '| seconds > 4h:', (raw.trip_seconds > 4*3600).sum())
print()
print('Most extreme rows by trip_total:')
print(raw.nlargest(6, 'trip_total')[['trip_seconds','trip_miles','fare','tips','tolls','extras','trip_total','payment_type']])

In [ ]:
# Does trip_total equal fare + tips + tolls + extras?
gap = (raw.trip_total - raw[['fare','tips','tolls','extras']].sum(axis=1, skipna=False)).round(2)
print('Rows compared:', gap.notna().sum())
print(gap.value_counts().sort_index().rename('rows').to_frame().T.to_string()); print()
print('Rows where trip_total < sum of parts:', (gap < -0.005).sum(), '| largest gap:', gap.max())
print('Share of rows with trip_total > sum of parts, by payment type:')
print(raw.assign(over=gap > 0).groupby('payment_type').over.mean().mul(100).round(1).to_frame('% rows with gap'))
print()
print('gap of $2.50 / $3.00 rows, top pickup areas:', raw.loc[gap.isin([2.5, 3.0]), 'pickup_community_area'].value_counts().head(4).to_dict())

**Reading the checks**

- `trip_total` is **never lower** than the sum of its parts. In 48,197 rows (35.5%) it is higher, always by at most $3.00. The gap is $0.50 in 46,522 rows and is almost entirely on Credit Card and Mobile trips, so it looks like a card/processing charge that is not itemised in `fare`, `tips`, `tolls` or `extras`. The $2.50 and $3.00 gaps (1,555 rows) cluster in a few pickup areas and look like a location surcharge. 
- **Assumption:** `trip_total` is what the rider was billed, so I use it as revenue as reported and **do not** rebuild it from its parts. Rows with a small gap are kept; no gap is large enough to suggest corruption.
- The 324 rows with missing money fields are 99.7% Cash, with all five fields missing together, so revenue and tip cannot be recovered.
- Distance and duration have extreme values (e.g. 793 miles in 51 minutes, 23-hour trips) and 2,637 trips with zero miles and zero seconds (most are a $3.25 flag-drop fare). **None of the four questions uses miles or duration**, so those fields are reported here but not cleaned, and nothing is excluded for them.

In [ ]:
# Exclusion rules, applied in order; each row is counted under the first rule it meets, so the counts add up.
rules = [
    ('R1  Missing fare/tips/tolls/extras/trip_total',   raw.trip_total.isna()),
    ('R2  trip_total = 0 (no revenue; fare is also 0)',   raw.trip_total.fillna(1) <= 0),
    ('R3  Extreme charge: trip_total > $500 or tolls/extras > $100',
        (raw.trip_total > 500) | (raw.tolls > 100) | (raw.extras > 100)),
]
remaining = pd.Series(True, index=raw.index)
log = []
for name, flag in rules:
    hit = flag & remaining
    log.append((name, int(hit.sum()), raw.loc[hit, 'trip_total'].sum()))
    remaining &= ~flag
log = pd.DataFrame(log, columns=['rule', 'rows_excluded', 'trip_total_removed($)'])
log.loc[len(log)] = ['Total excluded', log.rows_excluded.sum(), log['trip_total_removed($)'].sum()]
display(log)

df = raw[remaining].copy()
print(f'Rows kept: {len(df):,} of {len(raw):,} ({len(df)/len(raw):.2%})')

**Cleaning decisions (all applied to every question below)**

| # | Decision | Rows affected |
|---|---|---|
| R1 | Drop trips with no money fields: no fare, tip or total to analyse. | 324 |
| R2 | Drop trips whose `trip_total` is $0 (fare also $0): no revenue, likely voided or test records. | see log above |
| R3 | Drop clearly erroneous charges: `trip_total` > $500 or tolls/extras > $100 (includes tolls/extras of $4,444.44 on a short trip and an $8,901 total). | see log above |
| – | Keep the $0.50–$3.00 gap between `trip_total` and its parts (see above). | 48,197 kept |
| – | Keep trips with zero miles/seconds and long durations: miles and duration are not used. | not excluded |
| – | Keep rows with no pickup community area but report their share explicitly (Q2). | kept |
| – | Treat `Unknown`, `No Charge` and `Dispute` as payment types in their own right (not dropped). | kept |
| – | Merge two company names that differ only in punctuation or a legal suffix (below). | see below |

I did **not** drop a further 155 trips with a fare over $100 but 0 miles, because a fare alone is not proof of error. A sensitivity check follows to show it would not change the picture.

In [ ]:
# Company names: same company spelled two ways. Merge only the unambiguous cases.
variants = {'Taxicab Insurance Agency, LLC': 'Taxicab Insurance Agency Llc',
            'Choice Taxi Association Inc':   'Choice Taxi Association'}
print('Rows renamed:', df.company.isin(variants).sum(), '|', df.company[df.company.isin(variants)].value_counts().to_dict())
df['company'] = df.company.replace(variants)
print('Distinct companies:', raw.company.nunique(), '->', df.company.nunique())

# Do taxi_ids appear under more than one company?
print('taxi_ids under >1 company:', (df.groupby('taxi_id').company.nunique() > 1).sum(), 'of', df.taxi_id.nunique())

# Sensitivity: effect of the 155 'fare > $100 and 0 miles' trips that were kept
sus = (df.fare > 100) & (df.trip_miles == 0)
print(f'Kept trips with fare>$100 and 0 miles: {sus.sum()} = {sus.mean():.2%} of trips, {df.loc[sus,"trip_total"].sum()/df.trip_total.sum():.2%} of revenue')

# Time fields: derive day and hour from the (15-minute rounded) start timestamp
df['date'] = df.trip_start_timestamp.dt.normalize()
df['dow']  = df.trip_start_timestamp.dt.dayofweek          # Monday = 0
df['hour'] = df.trip_start_timestamp.dt.hour
df['is_weekend'] = df.dow >= 5

Notes on time: start timestamps are rounded to the nearest 15 minutes, so a trip beginning at 08:53 is stamped 09:00 and lands in the 09:00 hour. Hourly counts are therefore accurate to roughly ±7 minutes at the hour boundaries. All times are as given (local Chicago time is assumed; `dataset.md` does not state a time zone).

## 3. Q1 — When is demand highest?

In [ ]:
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
by_day = df.groupby('dow').size().reindex(range(7))
by_day.index = days
by_day_tbl = by_day.to_frame('trips').assign(share_pct=lambda d: d.trips / d.trips.sum() * 100)
display(by_day_tbl)

fig, ax = plt.subplots(figsize=(7, 3.6))
cols = [BLUE]*5 + [ORANGE]*2
bars = ax.bar(days, by_day.values, color=cols, width=0.65)
ax.bar_label(bars, labels=[f'{v:,}' for v in by_day.values], padding=3, fontsize=9)
ax.set_title('Reported trips by day of week, 11–17 Sep 2023')
ax.set_xlabel('Day of week (blue = weekday, orange = weekend)'); ax.set_ylabel('Trips')
ax.yaxis.set_major_formatter(thousands); ax.set_ylim(0, by_day.max() * 1.12)
plt.tight_layout(); plt.show()

In [ ]:
# Hour of day: average trips per hour on a weekday (Mon-Fri) vs a weekend day (Sat-Sun)
n_days = df.groupby('is_weekend').date.nunique()
hourly = (df.groupby(['is_weekend', 'hour']).size().unstack(0).reindex(range(24), fill_value=0)
            .rename(columns={False: 'Weekday', True: 'Weekend'}))
hourly_avg = hourly / n_days.rename({False: 'Weekday', True: 'Weekend'})
hourly_avg['Total trips (week)'] = hourly.sum(axis=1)

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(hourly_avg.index, hourly_avg['Weekday'], color=BLUE, lw=2, marker='o', ms=4, label='Weekday (Mon–Fri, avg per day)')
ax.plot(hourly_avg.index, hourly_avg['Weekend'], color=ORANGE, lw=2, marker='o', ms=4, label='Weekend (Sat–Sun, avg per day)')
ax.set_title('Average trips per hour of start time: weekdays vs weekends')
ax.set_xlabel('Hour of trip start (0 = midnight; timestamps rounded to 15 min)'); ax.set_ylabel('Average trips per day in that hour')
ax.set_xticks(range(0, 24, 2)); ax.yaxis.set_major_formatter(thousands)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

# Day x hour heatmap
grid = df.groupby(['dow', 'hour']).size().unstack().reindex(index=range(7), columns=range(24), fill_value=0)
grid.index = days
fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(grid.values, aspect='auto', cmap='Blues')
ax.set_yticks(range(7)); ax.set_yticklabels(days); ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))
ax.grid(False)
ax.set_title('Trips by day and hour of start'); ax.set_xlabel('Hour of trip start (0 = midnight)'); ax.set_ylabel('Day of week')
cb = fig.colorbar(im, ax=ax, pad=0.01); cb.set_label('Trips in that hour')
plt.tight_layout(); plt.show()

In [ ]:
wk, we = hourly_avg['Weekday'], hourly_avg['Weekend']
tot_hr = hourly['Weekday'] + hourly['Weekend']
print('Busiest day:', by_day.idxmax(), f'({by_day.max():,})', '| quietest day:', by_day.idxmin(), f'({by_day.min():,})')
print(f'Mon–Thu average: {by_day[:4].mean():,.0f} | Fri: {by_day["Fri"]:,} | Sat–Sun average: {by_day[5:].mean():,.0f} ({by_day[5:].mean()/by_day[:4].mean()-1:+.0%} vs Mon–Thu)')
print('Peak hour, weekday:', wk.idxmax(), f'({wk.max():,.0f}/day) | weekend:', we.idxmax(), f'({we.max():,.0f}/day)')
print('Quietest hour, weekday:', wk.idxmin(), f'({wk.min():,.0f}/day) | weekend:', we.idxmin(), f'({we.min():,.0f}/day)')
print('Top 5 hours of the week overall:', tot_hr.nlargest(5).to_dict())
print('Busiest single day-hour cell:', grid.stack().idxmax(), grid.stack().max(), '| quietest:', grid.stack().idxmin(), grid.stack().min())
print('Share of trips starting 3–6pm weekday:', f'{hourly.loc[15:17,"Weekday"].sum()/hourly["Weekday"].sum():.1%}')
print('Weekday ratio evening (18-23h) vs weekend evening:', round(wk.loc[18:23].sum()/we.loc[18:23].sum(), 2))
display(hourly_avg.round(0).astype(int).T)

**Takeaway (Q1)**: PLACEHOLDER_Q1

## 4. Q2 — Where do trips start?

Geography uses pickup **community area** codes (1–77), as asked. Names are not added because outside data is out of scope.

In [ ]:
pa = df.groupby('pickup_community_area').agg(trips=('trip_id', 'size'), total_revenue=('trip_total', 'sum'))
pa.index = pa.index.astype(int); pa.index.name = 'community_area'
pa['trip_share_pct'] = pa.trips / len(df) * 100
pa['revenue_share_pct'] = pa.total_revenue / df.trip_total.sum() * 100

top_trips = pa.nlargest(10, 'trips').assign(rank_trips=range(1, 11))
top_rev = pa.nlargest(10, 'total_revenue').assign(rank_revenue=range(1, 11))
print('Top 10 pickup community areas by trip count'); display(top_trips[['rank_trips','trips','trip_share_pct','total_revenue']])
print('Top 10 pickup community areas by total trip_total ($)'); display(top_rev[['rank_revenue','total_revenue','revenue_share_pct','trips']])
print('Areas in both lists:', len(set(top_trips.index) & set(top_rev.index)), '| in one only:', sorted(set(top_trips.index) ^ set(top_rev.index)))
print('Top 10 by trips cover', f'{top_trips.trips.sum()/len(df):.1%}', 'of all trips;', 'top 3 cover', f'{top_trips.trips.head(3).sum()/len(df):.1%}')

no_area = df.pickup_community_area.isna()
print(f'\nTrips with no pickup community area: {no_area.sum():,} = {no_area.mean():.2%} of {len(df):,} trips '
      f'(raw file before cleaning: {raw.pickup_community_area.isna().mean():.2%}); revenue share {df.loc[no_area,"trip_total"].sum()/df.trip_total.sum():.2%}')
print('  ...of which have no pickup latitude/longitude either:', f'{df.loc[no_area,"pickup_centroid_latitude"].isna().mean():.1%}',
      '| and no pickup census tract:', f'{df.loc[no_area,"pickup_census_tract"].isna().mean():.1%}')
print('  Their average trip_total: $%.2f vs $%.2f for trips with an area; average miles %.1f vs %.1f' % (
      df.loc[no_area,'trip_total'].mean(), df.loc[~no_area,'trip_total'].mean(), df.loc[no_area,'trip_miles'].mean(), df.loc[~no_area,'trip_miles'].mean()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, tbl, col, title, xlabel, fmt in [
    (axes[0], top_trips, 'trips', 'Top 10 pickup areas by trips', 'Trips', thousands),
    (axes[1], top_rev, 'total_revenue', 'Top 10 pickup areas by total trip_total', 'Total trip_total ($)', dollars)]:
    t = tbl.sort_values(col)
    b = ax.barh(t.index.astype(str), t[col], color=BLUE, height=0.65)
    ax.bar_label(b, labels=[f'{v:,.0f}' if col=='trips' else f'${v:,.0f}' for v in t[col]], padding=3, fontsize=8.5)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel('Pickup community area (code)')
    ax.xaxis.set_major_formatter(fmt); ax.set_xlim(0, t[col].max() * 1.18); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

**Takeaway (Q2)**: PLACEHOLDER_Q2

## 5. Q3 — How do riders pay and tip?

Definitions: *revenue* = sum of `trip_total`; *average fare* = mean `fare`; *tip % of fare* = sum of `tips` ÷ sum of `fare` (an aggregate ratio, so zero-fare and tiny-fare trips cannot blow up a per-trip percentage). *% trips with a tip* = share of trips with `tips` > 0. Cash tips are generally not recorded, so tip figures for non-card types are **not** comparable with cards.

In [ ]:
pay = df.groupby('payment_type').agg(trips=('trip_id', 'size'), total_revenue=('trip_total', 'sum'),
                                     avg_fare=('fare', 'mean'), fare_sum=('fare', 'sum'), tips_sum=('tips', 'sum'),
                                     pct_trips_with_tip=('tips', lambda s: (s > 0).mean() * 100))
pay['trip_share_pct'] = pay.trips / pay.trips.sum() * 100
pay['revenue_share_pct'] = pay.total_revenue / pay.total_revenue.sum() * 100
pay['tip_pct_of_fare'] = pay.tips_sum / pay.fare_sum * 100
pay = pay.sort_values('trips', ascending=False)
cols = ['trips', 'trip_share_pct', 'total_revenue', 'revenue_share_pct', 'avg_fare', 'tips_sum', 'tip_pct_of_fare', 'pct_trips_with_tip']
tot = pd.Series({'trips': pay.trips.sum(), 'trip_share_pct': 100.0, 'total_revenue': pay.total_revenue.sum(), 'revenue_share_pct': 100.0,
                 'avg_fare': df.fare.mean(), 'tips_sum': pay.tips_sum.sum(), 'tip_pct_of_fare': pay.tips_sum.sum()/pay.fare_sum.sum()*100,
                 'pct_trips_with_tip': (df.tips > 0).mean() * 100}, name='All trips')
display(pd.concat([pay[cols], tot.to_frame().T])
        .style.format({'trips': '{:,.0f}', 'total_revenue': '${:,.0f}', 'avg_fare': '${:.2f}', 'tips_sum': '${:,.0f}', 'trip_share_pct': '{:.1f}%',
                       'revenue_share_pct': '{:.1f}%', 'tip_pct_of_fare': '{:.1f}%', 'pct_trips_with_tip': '{:.1f}%'}))

# Like-for-like tipping: only the two card-based types where tips are recorded
card = df[df.payment_type.isin(['Credit Card', 'Mobile'])]
c = card.groupby('payment_type').apply(lambda g: pd.Series({'trips': len(g), 'tip_pct_of_fare': g.tips.sum()/g.fare.sum()*100,
        'median_tip_pct_of_fare (tipped trips, fare>0)': (g.tips/g.fare)[(g.tips > 0) & (g.fare > 0)].median()*100,
        '% trips tipped': (g.tips > 0).mean()*100}), include_groups=False)
print('Tipping on Credit Card vs Mobile only (tips recorded for both):'); display(c.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
order = pay.sort_values('trips').index
b = axes[0].barh(order, pay.loc[order, 'trips'], color=BLUE, height=0.65)
axes[0].bar_label(b, labels=[f'{v:,.0f}' for v in pay.loc[order, 'trips']], padding=3, fontsize=8.5)
axes[0].set_title('Trips by payment type'); axes[0].set_xlabel('Trips'); axes[0].set_ylabel('Payment type')
axes[0].xaxis.set_major_formatter(thousands); axes[0].set_xlim(0, pay.trips.max()*1.18)
b = axes[1].barh(order, pay.loc[order, 'total_revenue'], color=BLUE, height=0.65)
axes[1].bar_label(b, labels=[f'${v:,.0f}' for v in pay.loc[order, 'total_revenue']], padding=3, fontsize=8.5)
axes[1].set_title('Revenue (sum of trip_total) by payment type'); axes[1].set_xlabel('Revenue ($)'); axes[1].set_ylabel('Payment type')
axes[1].xaxis.set_major_formatter(dollars); axes[1].set_xlim(0, pay.total_revenue.max()*1.2)
for a in axes: a.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

**Takeaway (Q3)**: PLACEHOLDER_Q3

## 6. Q4 — Which companies carry the trips?

`taxi_id` counts are distinct taxis seen in this week's reported trips for that company. Two company names that differed only by punctuation/legal suffix were merged (section 2); no `taxi_id` appears under more than one company.

In [ ]:
co = df.groupby('company').agg(trips=('trip_id', 'size'), total_revenue=('trip_total', 'sum'), distinct_taxis=('taxi_id', 'nunique'))
co['share_of_all_trips_pct'] = co.trips / len(df) * 100
co['revenue_per_trip'] = co.total_revenue / co.trips
co['trips_per_taxi'] = co.trips / co.distinct_taxis
top_co = co.nlargest(10, 'trips')
top_co.index.name = 'company'
display(top_co.style.format({'trips': '{:,.0f}', 'total_revenue': '${:,.0f}', 'distinct_taxis': '{:,.0f}', 'share_of_all_trips_pct': '{:.1f}%',
                             'revenue_per_trip': '${:.2f}', 'trips_per_taxi': '{:.1f}'}))
print(f'Top 10 companies: {top_co.trips.sum()/len(df):.1%} of trips, {top_co.total_revenue.sum()/df.total_revenue.sum() if False else top_co.total_revenue.sum()/df.trip_total.sum():.1%} of revenue, '
      f'{top_co.distinct_taxis.sum():,} taxis of {df.taxi_id.nunique():,}. Top 3: {top_co.trips.head(3).sum()/len(df):.1%} of trips.')
print('Companies in total:', len(co), '| Next company after the top 10:', co.nlargest(11, 'trips').index[-1], f'({co.trips.nlargest(11).iloc[-1]:,} trips)')

fig, ax = plt.subplots(figsize=(8.5, 4.4))
t = top_co.sort_values('trips')
b = ax.barh(t.index, t.trips, color=BLUE, height=0.65)
ax.bar_label(b, labels=[f'{n:,.0f}  ({s:.1f}%)' for n, s in zip(t.trips, t.share_of_all_trips_pct)], padding=3, fontsize=8.5)
ax.set_title('Top 10 companies by reported trips (share of all trips in brackets)')
ax.set_xlabel('Trips'); ax.set_ylabel('Company'); ax.xaxis.set_major_formatter(thousands)
ax.set_xlim(0, t.trips.max()*1.25); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

**Takeaway (Q4)**: PLACEHOLDER_Q4

## 7. Assumptions, limitations and follow-up

- All figures are **reported** trips, not the whole market. Companies may differ in how completely they report, so company shares are shares of reported trips.
- Revenue is the sum of the reported `trip_total` (a gross billed amount, including tips, tolls and extras); it is not company earnings.
- One week, so no comparison with other weeks; Monday–Sunday patterns may not generalise to other times of year.
- Time zone is assumed to be local Chicago time. Hourly counts are blurred by the 15-minute timestamp rounding.
- Tips: cash tips are not recorded, and `Prcard`/`Unknown`/others show almost no tips, so tip rates are only meaningful for Credit Card and Mobile.
- **Possible follow-up:** PLACEHOLDER_FOLLOWUP